# Week 4 — Fine-Tuning a Language Model

**LLMs & You · Hampden-Sydney College · Fall 2026**

Every model you have used so far arrived finished. This notebook is where you
change one — take `distilgpt2`, show it a pile of Wikipedia, and measure whether
it got better at predicting Wikipedia.

Nothing here is prompting. The weights actually move.

---

## This is not a worksheet

There are no steps to complete and no correct output to reach. The code below
works. Running it top to bottom takes about fifteen minutes and teaches you very
little.

**What you are doing is looking for something to show on Thursday.** Change the
dataset, change the model, change how long it trains, and find something you did
not expect. A method that broke is worth more than one that worked.

There *is* something to hand in: six questions at the end, uploaded to Canvas
before class. They ask what you found while poking at the **Your turn** prompts
scattered through the notebook, so read those as you go rather than saving them
all for the end.

!!! warning "Turn the GPU on first, or this will take hours"

    **Runtime → Change runtime type → T4 GPU**, then run the setup cell.

    Everything below runs on a CPU too, but training is tens of times slower
    there. The setup cell prints which one you got. If it says `none`,
    you either forgot to switch or you have used up today's free GPU quota —
    in which case drop `TRAIN_ROWS` in section 2 to about 200 and carry on with
    a worse model, which is still a model.

    Nothing in this notebook needs a Hugging Face account, a token, or a card.

---
## 0. Setup

One cell, once per session. Colab forgets everything when the tab is closed, so
if you come back tomorrow, run this again.

In [ ]:
# Colab has torch, numpy and pandas already. These three are the ones it does not
# reliably have, and `accelerate` is the one whose absence is most confusing:
# without it, Trainer fails several cells later with an import error.
%pip install --quiet "transformers>=4.46" "datasets>=3.0" "accelerate>=1.1"

import math

import pandas as pd
import torch
import transformers

pd.set_option("display.max_colwidth", 200)

print("transformers", transformers.__version__)
print("torch       ", torch.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

---
## 1. A corpus the model has already half-seen

**WikiText-2** is about two million words of Wikipedia, chosen years ago as a
standard benchmark for exactly this task. It is small enough to train on in a
classroom and real enough that the numbers mean something.

There is a wrinkle worth noticing before you run the next cell.

!!! note "Almost every tutorial you will find calls this dataset by the wrong name"

    The published version of this lab — Hugging Face's own — says:

    ```python
    load_dataset("wikitext", "wikitext-2-raw-v1")
    ```

    That now fails: `Repository id must be 'namespace/name', got 'wikitext'`.
    Datasets on the Hub used to have bare names; they were moved under the
    accounts that own them, and `wikitext` became `Salesforce/wikitext`.

    You will meet this again. When a cell fails on a line you copied
    verbatim from a tutorial, the tutorial is the thing to doubt first --
    libraries move and published notebooks do not.

In [ ]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
raw

In [ ]:
# What is actually in here? Not clean prose -- headings, blank lines, and the
# tokenizer artefacts of whoever built this in 2016 (` @-@ ` was a hyphen).
sample = raw["train"].select(range(10, 20))
pd.DataFrame(sample)

!!! tip "Your turn"

    - Roughly what fraction of the training rows are empty or just a heading?
      Does that matter for what the model learns?
    - `' @-@ '` appears where a hyphen used to be. The model will learn to
      produce it. Is that a bug in the dataset, the model, or the benchmark?
    - Swap in a different dataset from
      [the Hub](https://huggingface.co/datasets) — or your own text file — and
      everything below still runs. This is the single highest-value change you
      can make to this notebook.

---
## 2. Cutting the text into training examples

A causal language model learns one thing: given the tokens so far, predict the
next one. To train it you need fixed-length blocks of tokens, so the text gets
tokenized, concatenated end to end, and sliced into chunks of `BLOCK_SIZE`.

Chunks therefore run across article boundaries. That is deliberate and it is why
the labels are simply a copy of the inputs — the model shifts them internally.

In [ ]:
from datasets import DatasetDict
from transformers import AutoTokenizer

MODEL = "distilgpt2"
BLOCK_SIZE = 128     # distilgpt2 can take 1024; 128 trains far faster and shows the same thing
TRAIN_ROWS = 4000    # of ~36,700. Raise this if you have a GPU and patience.
EVAL_ROWS = 1000

tokenizer = AutoTokenizer.from_pretrained(MODEL)

subset = DatasetDict({
    "train": raw["train"].select(range(TRAIN_ROWS)),
    "validation": raw["validation"].select(range(EVAL_ROWS)),
})


def tokenize(batch):
    return tokenizer(batch["text"])


tokenized = subset.map(tokenize, batched=True, remove_columns=["text"])


def group_texts(batch):
    joined = {k: sum(batch[k], []) for k in batch}
    total = (len(joined["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE  # drop the remainder
    out = {
        k: [v[i : i + BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)]
        for k, v in joined.items()
    }
    out["labels"] = out["input_ids"].copy()
    return out


lm = tokenized.map(group_texts, batched=True, batch_size=1000)
print(lm)
print("\nOne training example, decoded back to text:\n")
print(tokenizer.decode(lm["train"][1]["input_ids"]))

!!! tip "Your turn"

    - Find a block that starts mid-sentence in one article and ends inside the
      next. What is the model being asked to predict at that seam?
    - Raise `BLOCK_SIZE` to 512 and re-run. Fewer, longer examples: does the
      perplexity in section 4 get better or worse, and is that a fair comparison?
    - `group_texts` throws away the remainder of every 1,000-row batch. How much
      text is that, and how would you check rather than guess?

---
## 3. Measuring the model before you touch it

The number here is **perplexity** — roughly, how many equally-likely next tokens
the model feels it is choosing between. Lower is better. A perplexity of 50 means
the model is about as uncertain as if it were picking uniformly from 50 options.

Measure it *first*. A fine-tuning result with no before-number is not a result.

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained(MODEL)

args = TrainingArguments(
    output_dir="distilgpt2-wikitext2",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none",      # no experiment tracker, no account
    push_to_hub=False,     # the published version pushes to the Hub; that needs a token
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm["train"],
    eval_dataset=lm["validation"],
)

before = trainer.evaluate()
print(f"Perplexity before fine-tuning: {math.exp(before['eval_loss']):.2f}")

!!! note "`eval_strategy`, not `evaluation_strategy`"

    Older code says `evaluation_strategy=`. That argument was renamed and the
    old spelling is now a `TypeError`. Same lesson as the dataset name: the
    reference is current, the tutorial is not.

---
## 4. Actually training it

One pass over the subset. On a T4 this is a couple of minutes; on a CPU, go and
make coffee.

Watch the training loss as it goes. It should fall, unevenly.

In [ ]:
trainer.train()

after = trainer.evaluate()
print(f"Perplexity before: {math.exp(before['eval_loss']):.2f}")
print(f"Perplexity after:  {math.exp(after['eval_loss']):.2f}")

!!! tip "Your turn"

    - Perplexity roughly halves. Is that the model learning English, or the
      model learning the *format* of this particular dump -- the spaced
      punctuation, the headings, the ` @-@ `? Those are very different claims
      and section 5 is where you can tell them apart.
    - Raise `learning_rate` to `5e-4` and re-run from section 3. The loss does
      something dramatic. Explain it.
    - Train on `TRAIN_ROWS = 400` instead. How much worse? Less than you expect?

---
## 5. What the number does not show you

Perplexity went down. That is a claim about probability, not about writing. Look
at what the two models actually produce from the same prompt.

In [ ]:
from transformers import pipeline

DEVICE = trainer.args.device

# `model` has been fine-tuned in place, so the "before" model is loaded fresh.
original = pipeline("text-generation", model=MODEL, tokenizer=tokenizer, device=DEVICE)
finetuned = pipeline("text-generation", model=model, tokenizer=tokenizer, device=DEVICE)

PROMPT = "The history of the town is"

for name, pipe in [("original ", original), ("finetuned", finetuned)]:
    out = pipe(PROMPT, max_new_tokens=60, do_sample=True, temperature=0.8, top_p=0.95,
               truncation=True, pad_token_id=tokenizer.eos_token_id)
    print(f"--- {name} ---\n{out[0]['generated_text']}\n")

!!! note "That cell prints warnings. They are noise."

    `transformers` complains that `max_new_tokens` and `max_length` are both
    set, and that passing generation arguments alongside a `generation_config`
    is deprecated. Both come from `distilgpt2` shipping a default generation
    config that we are overriding, which is exactly what we want. The output
    below them is correct.

    Getting used to reading a warning and deciding it does not apply is part of
    the skill. The ones that matter name a thing you did.

!!! tip "Your turn — this is where the interesting findings are"

    - Run that cell five times. Is the fine-tuned model *better*, or just more
      Wikipedia-shaped? Those are different claims.
    - Look at the punctuation the fine-tuned model produces -- spaces before
      commas, and perhaps a ` @-@ `. It has learned a formatting artefact of
      this dump rather than anything about language. How much of the perplexity
      improvement you just measured do you think that accounts for?
    - Find a prompt where fine-tuning made the output clearly **worse**. That is
      a better thing to bring on Thursday than a prompt where it helped.

---
## 6. The other kind of language model

`distilgpt2` predicts the next token. `distilroberta-base` predicts a *missing*
token, with the whole sentence either side of it visible — masked language
modelling, the objective BERT introduced and the one behind the encoder half of
the transformer you looked at in Week 3.

The pipeline is the same except for one piece: a collator that hides 15% of the
tokens fresh every epoch, so the model never sees the same mask twice.

In [ ]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

MLM_MODEL = "distilroberta-base"
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_MODEL)

mlm_tokenized = subset.map(
    lambda b: mlm_tokenizer(b["text"]), batched=True, remove_columns=["text"]
)
mlm_lm = mlm_tokenized.map(group_texts, batched=True, batch_size=1000)

collator = DataCollatorForLanguageModeling(tokenizer=mlm_tokenizer, mlm_probability=0.15)

mlm_model = AutoModelForMaskedLM.from_pretrained(MLM_MODEL)
mlm_trainer = Trainer(
    model=mlm_model,
    args=TrainingArguments(
        output_dir="distilroberta-wikitext2",
        eval_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        report_to="none",
        push_to_hub=False,
    ),
    train_dataset=mlm_lm["train"],
    eval_dataset=mlm_lm["validation"],
    data_collator=collator,
)

mlm_before = mlm_trainer.evaluate()
mlm_trainer.train()
mlm_after = mlm_trainer.evaluate()

print(f"MLM perplexity before: {math.exp(mlm_before['eval_loss']):.2f}")
print(f"MLM perplexity after:  {math.exp(mlm_after['eval_loss']):.2f}")

!!! note "`Token indices sequence length is longer than ... (638 > 512)`"

    That warning fires while tokenizing, because some rows of Wikipedia are
    longer than `distilroberta-base` can take in one go. It would matter if we
    fed those rows to the model as they are. We do not: `group_texts` runs next
    and re-cuts everything into blocks of 128.

    A warning about a problem the following cell removes.

!!! warning "Do not compare these two perplexities to the ones in section 4"

    The masked model's perplexity is much lower, and it is **not** a better
    model. It is being asked an easier question: guess 15% of the tokens while
    looking at the other 85%. The causal model guesses every token having seen
    only what came before.

    Two numbers with the same name, measuring different tasks. This is the most
    common way a benchmark comparison quietly lies, and you will meet it again
    in Week 7 when you build an evaluation of your own.

In [ ]:
# What the masked model does that the causal one cannot: fill a gap using context
# from both sides.
filler = pipeline("fill-mask", model=mlm_model, tokenizer=mlm_tokenizer,
                  device=mlm_trainer.args.device)

for guess in filler(f"The town was founded in {mlm_tokenizer.mask_token} by settlers.", top_k=5):
    print(f"{guess['score']:.3f}  {guess['token_str']!r}")

!!! tip "Your turn"

    - Ask it to fill a mask where the answer is only knowable from the words
      *after* the gap. Does it get it? Try the same sentence on `distilgpt2`
      and watch it be structurally unable to.
    - The five candidates come with probabilities. Are they calibrated — does a
      0.6 mean it is right about 60% of the time? Design a check.

---
## 7. Write-up — six questions, due on Canvas before class

The **Your turn** prompts above are where the thinking happens; this is where
you write down what came of it. Answer all six **in full sentences** — not
bullets, not fragments — and upload the file to Canvas before Thursday's
session. A paragraph each is plenty; six good paragraphs beat two pages.

Answer from what you actually ran. "I did not get to this one" is an acceptable
answer to any single question and costs you nothing; inventing a result you did
not measure is the only way to do badly here.

**1. The corpus.** Look back at section 1. What did you notice about WikiText-2
that you would not have noticed if you had only read its description — the empty
rows, the headings, the spaced punctuation, the ` @-@ `? Name one specific thing
and say what you think it does to a model trained on it.

**2. The measurement.** Perplexity fell from roughly 78 to roughly 41 in section
4, or to whatever it fell to for you. State your two numbers. Do you believe the
model got better at English, better at Wikipedia, or better at reproducing the
formatting of this particular dump — and what in the notebook makes you say so?

**3. Where the number stopped helping.** In section 5 you read what the two
models actually wrote. Describe one concrete difference between the original and
fine-tuned output. Was the fine-tuned model *better*, or only more
Wikipedia-shaped? Those are different claims and this question is asking you to
commit to one.

**4. Two numbers with the same name.** Section 6's masked model scored a much
lower perplexity than the causal model and is not a better model. Explain, in
your own words, why comparing those two numbers would be a mistake. Then name
one place outside this notebook where you think you have seen that mistake made.

**5. A task worth fine-tuning.** Pick one task — something you would actually
use, from your own work, a club, a job, a hobby, your project for this course.
Describe it in two or three sentences. Then answer the harder half: why would
fine-tuning a model suit this task better than writing a careful prompt for a
model that already exists? If you conclude that prompting would in fact do the
job, say that and say why. That is a perfectly good answer and it is the right
one more often than not.

**6. Where the corpus would come from.** For the task you chose in question 5,
think through the data. Where would the examples come from? Roughly how many
could you realistically gather, and how long would collecting them take you?
Who owns that text, and are you allowed to train on it? Is there anything in it
— names, grades, messages, anything about a real person — that should not end up
inside a model's weights? Finish by saying whether you would still do it.

---
## What to bring on Thursday

Two different things, and it is worth keeping them apart. The six questions
above are written down and uploaded. What you bring to the room is **one
finding**, spoken — a claim plus the evidence you ran, in three sentences:

> *I expected X. I ran Y. I got Z, which surprised me because…*

A method that broke is worth more than one that worked — the labs are graded on
the write-up, not the result. A confusion you can state precisely is a finding.

Good places to have gone looking:

- A dataset that is not Wikipedia, and what fine-tuning on it did to the
  generated text.
- A learning rate that destroyed the model, and the loss curve that shows it.
- A case where perplexity improved and the output got worse.